# AIMS Senegal — Road Traffic Object Detection & Tracking
**Project 2 — Computer Vision | Deadline: April 28, 2026**

This notebook orchestrates the full pipeline: detection, tracking, and counting of road traffic objects across multiple video scenes.
All steps are called via CLI commands from this notebook.

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics kagglehub filterpy flask pandas plotly seaborn tqdm loguru

## 2. Clone Project from GitHub

In [ ]:
!git clone https://github.com/ioget/traffic-rookie-aims-cv.git
%cd traffic-rookie-aims-cv

## 3. Download BDD100K Dataset

In [ ]:
import kagglehub
import os

# Download latest versionB
path = kagglehub.dataset_download("a7madmostafa/bdd100k-yolo")

DATASET_PATH = path
original_yaml = f"{DATASET_PATH}/data.yaml"

# data.yaml has wrong paths — fix them and write to /tmp/ (always writable)
with open(original_yaml, 'r') as f:
    content = f.read()

content = content.replace('/kaggle/input/bdd100k-yolo', DATASET_PATH)

DATASET_YAML = "/tmp/data_fixed.yaml"
with open(DATASET_YAML, 'w') as f:
    f.write(content)

os.environ['DATASET_PATH'] = DATASET_PATH
os.environ['DATASET_YAML'] = DATASET_YAML

print("Dataset path :", DATASET_PATH)
print("Fixed YAML   :", DATASET_YAML)
print("\nFixed YAML content:")
print(content)

In [ ]:
# Explore dataset structure
!ls -lh {path}
!find {path} -name '*.yaml' | head -5

## 3b. Download VisDrone & Merge with BDD100K

In [ ]:
from ultralytics.utils.downloads import download
from pathlib import Path
import shutil, os

# Ultralytics downloads VisDrone automatically
VISDRONE_DIR = Path("/kaggle/working/VisDrone")
VISDRONE_DIR.mkdir(exist_ok=True)

urls = [
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
]

for url in urls:
    download(url, dir=VISDRONE_DIR)

print("VisDrone downloaded to:", VISDRONE_DIR)
!ls {VISDRONE_DIR}

In [ ]:
from pathlib import Path

# VisDrone → BDD100K class ID remapping
# VisDrone: 0=pedestrian,1=people,2=bicycle,3=car,4=van,5=truck,6=tricycle,7=awning-tricycle,8=bus,9=motor
# BDD100K:  0=person,1=rider,2=car,3=bus,4=truck,5=bike,6=motor,7=traffic light,8=traffic sign,9=train
VISDRONE_TO_BDD = {
    0: 0,   # pedestrian → person
    1: 1,   # people     → rider
    2: 5,   # bicycle    → bike
    3: 2,   # car        → car
    4: 4,   # van        → truck
    5: 4,   # truck      → truck
    8: 3,   # bus        → bus
    9: 6,   # motor      → motor
    # 6=tricycle, 7=awning-tricycle → skipped (no BDD equivalent)
}

MERGED_DIR = Path("/kaggle/working/merged_dataset")
for split in ["train", "val"]:
    (MERGED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

def remap_and_copy(src_img_dir, src_lbl_dir, split):
    src_img_dir = Path(src_img_dir)
    src_lbl_dir = Path(src_lbl_dir)
    copied = 0
    for img in src_img_dir.glob("*"):
        lbl = src_lbl_dir / img.with_suffix(".txt").name
        if not lbl.exists():
            continue
        lines = lbl.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            cls = int(parts[0])
            if cls in VISDRONE_TO_BDD:
                parts[0] = str(VISDRONE_TO_BDD[cls])
                new_lines.append(" ".join(parts))
        if new_lines:
            shutil.copy(img, MERGED_DIR / split / "images" / img.name)
            (MERGED_DIR / split / "labels" / lbl.name).write_text("\n".join(new_lines))
            copied += 1
    print(f"  {split}: {copied} files copied")

# Remap VisDrone labels and copy to merged dataset
visdrone_base = VISDRONE_DIR
print("Remapping VisDrone labels...")
for split, folder in [("train", "VisDrone2019-DET-train"), ("val", "VisDrone2019-DET-val")]:
    remap_and_copy(
        visdrone_base / folder / "images",
        visdrone_base / folder / "annotations",
        split
    )

# Copy BDD100K as-is (already correct class IDs)
print("Copying BDD100K...")
for split in ["train", "val"]:
    for img in Path(f"{DATASET_PATH}/{split}/images").glob("*"):
        shutil.copy(img, MERGED_DIR / split / "images" / img.name)
    for lbl in Path(f"{DATASET_PATH}/{split}/labels").glob("*"):
        shutil.copy(lbl, MERGED_DIR / split / "labels" / lbl.name)
    count = len(list((MERGED_DIR / split / "images").glob("*")))
    print(f"  {split}: {count} total images after merge")

# Write merged data.yaml
merged_yaml = "/tmp/data_merged.yaml"
Path(merged_yaml).write_text(f"""path: {MERGED_DIR}
train: {MERGED_DIR}/train/images
val: {MERGED_DIR}/val/images
nc: 10
names:
  - person
  - rider
  - car
  - bus
  - truck
  - bike
  - motor
  - traffic light
  - traffic sign
  - train
""")
DATASET_YAML = merged_yaml
os.environ['DATASET_YAML'] = DATASET_YAML
print(f"\nMerged YAML saved to {merged_yaml}")

## 4. Environment Check (GPU, Files)

In [ ]:
!python -c "import torch; print('GPU available:', torch.cuda.is_available()); print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!python -c "from ultralytics import YOLO; print('Ultralytics OK')"
!ls data/
!ls models/

## 5. Fine-tune YOLO on BDD100K

In [ ]:
from src.detection.yolo_detector import YOLODetector

detector = YOLODetector(model_path='models/yolo11n.pt', confidence_threshold=0.5)
detector.fine_tune_model(dataset_config=DATASET_YAML, epochs=10)
print('Fine-tuning complete.')

## 5b. Save Fine-tuned Model

In [ ]:
import shutil
import glob
import os

# Find the best weights produced by fine-tuning
best_weights = glob.glob("runs/detect/**/best.pt", recursive=True)

if best_weights:
    src = best_weights[0]
    dst = "models/yolo11n_finetuned.pt"
    shutil.copy(src, dst)
    print(f"Saved fine-tuned model to {dst}")
    print(f"File size: {os.path.getsize(dst) / 1e6:.1f} MB")
else:
    print("No best.pt found. Make sure fine-tuning completed successfully.")

In [ ]:
# Push fine-tuned model to GitHub
!git config user.email "mamakemrosly@gmail.com"
!git config user.name "ioget"
!git add models/yolo11n_finetuned.pt
!git commit -m "Add fine-tuned YOLOv11 model on BDD100K (10 epochs)"
!git push

## 6. Detection + Tracking — Video 1 (traffic.mp4)

In [ ]:
!python main.py \
    --video data/traffic.mp4 \
    --model models/yolo11n_finetuned.pt \
    --output outputs/traffic_finetuned.avi \
    --confidence 0.5 \
    --tracking

## 7. Detection + Tracking — Video 2 (traffic-sign-test.mp4)

In [ ]:
!python main.py \
    --video data/traffic-sign-test.mp4 \
    --model models/yolo11n_finetuned.pt \
    --output outputs/traffic_sign_finetuned.avi \
    --confidence 0.5 \
    --tracking

## 7b. Detection + Tracking — Video 3 (fecc9a75 traffic scene)

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

!python main.py \
    --video "data/fecc9a75-f0ac-49bc-8e48-02e71bef2cd7-0.mp4" \
    --model models/yolo11n_finetuned.pt \
    --output outputs/fecc9a75_finetuned.avi \
    --confidence 0.45 \
    --tracking

## 8. Check Generated Logs

In [ ]:
import json, glob

!ls -lh logs/

log_files = glob.glob('logs/*.json')
for f in log_files:
    with open(f) as fp:
        data = json.load(fp)
    detections = data.get('detections', data) if isinstance(data, dict) else data
    print(f"--- {f} ---")
    print(f"  Total entries: {len(detections)}")
    if detections:
        print(f"  Sample: {detections[0]}")

## 9. Aggregated Statistics

In [ ]:
import json, glob
from collections import Counter

all_detections = []
for f in glob.glob('logs/*.json'):
    with open(f) as fp:
        data = json.load(fp)
    detections = data.get('detections', data) if isinstance(data, dict) else data
    all_detections.extend(detections)

class_counts = Counter(d.get('class_name', 'unknown') for d in all_detections)
print('Total count per class:')
for cls, cnt in class_counts.most_common():
    print(f'  {cls}: {cnt}')
print(f'Total detections: {len(all_detections)}')

## 10. Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import json, glob
from collections import Counter

all_detections = []
for f in glob.glob('logs/*.json'):
    with open(f) as fp:
        data = json.load(fp)
    detections = data.get('detections', data) if isinstance(data, dict) else data
    all_detections.extend(detections)

class_counts = Counter(d.get('class_name', 'unknown') for d in all_detections)

if class_counts:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(class_counts.keys(), class_counts.values(), color='steelblue')
    ax.set_title('Detection Count per Object Class')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Detections')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('outputs/stats_classes.png', dpi=150)
    plt.show()
    print('Chart saved to outputs/stats_classes.png')
else:
    print('No log data found. Run steps 6 and 7 first.')

## 11. Web Interface (local use only)
> On Kaggle, the web interface is not directly accessible. Run this locally instead.

In [ ]:
# Uncomment below to launch locally
# !python main.py --video data/traffic.mp4 --model models/yolo11n.pt --web
print("To launch the web interface locally, run: python main.py --video data/traffic.mp4 --web")

## 12. Check Output Files

In [ ]:
!ls -lh outputs/ 2>/dev/null || echo 'outputs/ folder is empty or does not exist'
!ls -lh logs/ 2>/dev/null || echo 'logs/ folder is empty or does not exist'

## 13. Play Output Video

In [ ]:
from IPython.display import Video, display
import glob

output_files = glob.glob("outputs/*.avi") + glob.glob("outputs/*.mp4")

if output_files:
    for f in output_files:
        print(f"Playing: {f}")
        display(Video(f, embed=True, width=800))
else:
    print("No output videos found. Run steps 6 and 7 first.")